# Lab 10: Client APIs for DuckDB

## 🎯 Learning Objectives

- Understand officially supported DuckDB client languages
- Learn about concurrency considerations in DuckDB
- Explore different use cases for client APIs
- Practice importing large amounts of data efficiently
- Learn to use DuckDB from Java via JDBC Driver
- Understand the general usage pattern for JDBC
- Practice using multiple connections from several threads
- Use DuckDB as a data processing tool from Java
- Practice inserting large amounts of data from Java
- Explore additional connection options
- Compare different client API approaches

## 🎓 Conceptual Background

### Officially Supported Languages

DuckDB provides client libraries for multiple programming languages.

## 🚀 Step 1: Python API Mastery

Practice advanced Python API usage.

In [ ]:
import duckdb
import pandas as pd
import time

print("=== Python API Mastery ===")

# Connection management
print("\n1. Connection Management:")
con_memory = duckdb.connect(':memory:')
con_persistent = duckdb.connect('data/python_api_test.db')
con_readonly = duckdb.connect('data/duckdb_practice.db', read_only=True)
print("✓ Created in-memory, persistent, and read-only connections")

# Configuration options
print("\n2. Configuration Options:")
con_config = duckdb.connect(':memory:', config={
    'threads': 4,
    'memory_limit': '2GB',
    'default_order': 'ASC'
})
print("✓ Configured connection with custom settings")

# Performance testing
print("\n3. Performance Testing:")
con_config.execute("CREATE TABLE test AS SELECT * FROM range(1000000)")
start = time.time()
result = con_config.execute("SELECT SUM(column0) FROM test").fetchone()
perf_time = time.time() - start
print(f"✓ Performance: Query completed in {perf_time:.4f}s")

# Cleanup
con_memory.close()
con_persistent.close()
con_readonly.close()
con_config.close()

print("\n✓ Python API mastery demonstrated")

## 🚀 Step 2: Importing Large Amounts of Data

Practice efficient data import strategies.

In [ ]:
import numpy as np

print("=== Large Data Import Strategies ===")

con = duckdb.connect('data/large_import.db')

# Method 1: Batch insertion
print("\n1. Batch Insertion:")
start = time.time()
for i in range(10):
    batch_data = pd.DataFrame({
        'id': range(i * 10000, (i + 1) * 10000),
        'value': np.random.rand(10000) * 100,
        'category': np.random.choice(['A', 'B', 'C'], 10000)
    })
    con.register('batch', batch_data)
    con.execute("INSERT INTO large_table SELECT * FROM batch")
    con.execute("DROP TABLE batch")
batch_time = time.time() - start
print(f"✓ Batch insertion: {batch_time:.4f}s")

# Method 2: Appender (most efficient)
print("\n2. Appender Method:")
con.execute("CREATE TABLE large_table2 (id INTEGER, value DOUBLE, category VARCHAR)")
start = time.time()
appender = con.appender('large_table2')
for i in range(100000):
    appender.append_row(i, np.random.rand() * 100, np.random.choice(['A', 'B', 'C']))
appender.close()
appender_time = time.time() - start
print(f"✓ Appender: {appender_time:.4f}s")

print(f"\nPerformance comparison:")
print(f"Batch insertion: {batch_time:.4f}s")
print(f"Appender: {appender_time:.4f}s")
print(f"Appender is {batch_time/appender_time:.2f}x faster")

con.close()

## 🚀 Step 3: Java JDBC Framework

**Note**: Java JDBC requires Java installation and DuckDB JDBC driver.

This section provides the framework for Java JDBC integration.

In [ ]:
# Java JDBC framework code
java_jdbc_code = '''
import java.sql.*;

public class DuckDBJDBCBasic {
    public static void main(String[] args) {
        // Connection string
        String url = "jdbc:duckdb:data/java_test.db";
        
        try {
            // Establish connection
            Connection conn = DriverManager.getConnection(url);
            System.out.println("✓ Connected to DuckDB");
            
            // Create statement
            Statement stmt = conn.createStatement();
            
            // Create table
            stmt.execute("CREATE TABLE users (id INTEGER, name VARCHAR, email VARCHAR)");
            System.out.println("✓ Table created");
            
            // Insert data
            stmt.execute("INSERT INTO users VALUES (1, 'Alice', 'alice@example.com')");
            stmt.execute("INSERT INTO users VALUES (2, 'Bob', 'bob@example.com')");
            System.out.println("✓ Data inserted");
            
            // Query data
            ResultSet rs = stmt.executeQuery("SELECT * FROM users");
            while (rs.next()) {
                System.out.println("User: " + rs.getInt("id") + 
                                 ", " + rs.getString("name") + 
                                 ", " + rs.getString("email"));
            }
            
            // Cleanup
            rs.close();
            stmt.close();
            conn.close();
            System.out.println("✓ Connection closed");
            
        } catch (SQLException e) {
            e.printStackTrace();
        }
    }
}
'''

# Save Java code
with open('data/DuckDBJDBCBasic.java', 'w') as f:
    f.write(java_jdbc_code)

print("Java JDBC Framework:")
print("="*50)
print("✓ Java JDBC code saved to: data/DuckDBJDBCBasic.java")
print("\nTo use Java JDBC:")
print("1. Install Java 11+")
print("2. Download DuckDB JDBC driver from https://github.com/duckdb/duckdb/releases")
print("3. Compile: javac -cp duckdb.jar DuckDBJDBCBasic.java")
print("4. Run: java -cp duckdb.jar:. DuckDBJDBCBasic")

## 🚀 Step 4: Concurrency Patterns

Practice concurrent access patterns.

In [ ]:
import threading
import queue

print("=== Concurrency Patterns ===")

# Simulate concurrent readers
def concurrent_reader(db_path, result_queue, thread_id):
    """Simulate concurrent read operation"""
    try:
        con = duckdb.connect(db_path, read_only=True)
        result = con.execute("SELECT COUNT(*) FROM sample_customers").fetchone()[0]
        result_queue.put((thread_id, result))
        con.close()
    except Exception as e:
        result_queue.put((thread_id, f"Error: {e}"))

# Test concurrent reads
print("\n1. Concurrent Readers:")
result_queue = queue.Queue()
threads = []

for i in range(5):
    thread = threading.Thread(target=concurrent_reader, 
                               args=('data/duckdb_practice.db', result_queue, i))
    threads.append(thread)
    thread.start()

for thread in threads:
    thread.join()

print("Concurrent read results:")
while not result_queue.empty():
    thread_id, result = result_queue.get()
    print(f"  Thread {thread_id}: {result}")

# Test batched writes
print("\n2. Batched Writes:")
con = duckdb.connect('data/concurrency_test.db')
con.execute("CREATE TABLE test_concurrent (id INTEGER, value INTEGER)")

def batch_writer(con, start_id, batch_size):
    """Perform batched write operation"""
    for i in range(start_id, start_id + batch_size):
        con.execute(f"INSERT INTO test_concurrent VALUES ({i}, {i * 10})")

start_time = time.time()
batch_writer(con, 0, 1000)
batch_time = time.time() - start_time

count = con.execute("SELECT COUNT(*) FROM test_concurrent").fetchone()[0]
print(f"✓ Batched write: {count} rows in {batch_time:.4f}s")

con.close()

print("\n✓ Concurrency patterns demonstrated")

## 🚀 Step 5: Additional Connection Options

Explore advanced connection options.

In [ ]:
print("=== Advanced Connection Options ===")

# Thread-safe connection
print("\n1. Thread-safe Connection:")
con_threadsafe = duckdb.connect('data/threadsafe.db', config={
    'threads': 4,
    'memory_limit': '2GB',
    'default_order': 'ASC'
})
print("✓ Thread-safe connection with custom config")

# Read-only connection with specific access mode
print("\n2. Read-only Configuration:")
con_readonly_config = duckdb.connect('data/duckdb_practice.db', 
                                   read_only=True,
                                   config={'access_mode': 'read_only'})
print("✓ Configured read-only connection")

# In-memory with specific size limit
print("\n3. In-memory Size Limit:")
con_memory_limit = duckdb.connect(':memory:', config={
    'max_memory': '1GB'
})
print("✓ In-memory connection with size limit")

# WAL mode for better concurrency
print("\n4. WAL Mode Configuration:")
con_wal = duckdb.connect('data/wal_mode.db', config={
    'wal_autocheckpoint': '1GB'
})
print("✓ WAL mode connection")

# Display all configuration options
print("\n5. Configuration Options Summary:")
con_config = duckdb.connect(':memory:')
all_settings = con_config.execute("SELECT * FROM duckdb_settings()").fetchall()
print(f"Total configuration options: {len(all_settings)}")
print("\nKey options:")
key_options = ['threads', 'memory_limit', 'default_order', 'access_mode', 'max_memory', 'wal_autocheckpoint']
for setting in all_settings:
    if setting[0] in key_options:
        print(f"  {setting[0]}: {setting[1]}")

# Cleanup
con_threadsafe.close()
con_readonly_config.close()
con_memory_limit.close()
con_wal.close()
con_config.close()

print("\n✓ Advanced connection options demonstrated")

## 💻 Exercise 1: Python API Mastery

Practice advanced Python API usage.

In [ ]:
# Your advanced Python API code here:
# 1. Connection pooling
# 2. Transaction management
# 3. Error handling
# 4. Performance optimization

## 💻 Exercise 2: Concurrency Patterns

Implement concurrent access patterns.

In [ ]:
# Your concurrency implementation here:
# 1. Multi-threaded reads
# 2. Batched writes
# 3. Connection pooling
# 4. Conflict resolution

## ✅ Verification

Verify your API knowledge.

In [ ]:
print("=== Client API Verification ===")

# Test 1: Python API
con = duckdb.connect(':memory:')
con.execute("CREATE TABLE test AS SELECT * FROM range(100)")
result = con.execute("SELECT COUNT(*) FROM test").fetchone()
print(f"✓ Python API: {result[0]} rows")

# Test 2: Connection options
con_config = duckdb.connect(':memory:', config={'threads': 4})
config = con_config.execute("SELECT current_setting('threads')").fetchone()
print(f"✓ Connection config: {config[0]} threads")

# Test 3: Performance
start = time.time()
con.execute("SELECT SUM(column0) FROM test").fetchone()
perf = time.time() - start
print(f"✓ Performance: Query completed in {perf:.4f}s")

# Test 4: Java framework
import os
if os.path.exists('data/DuckDBJDBCBasic.java'):
    print("✓ Java JDBC framework created")
else:
    print("✗ Java JDBC framework not found")

con.close()
con_config.close()

print("=== Verification Complete ===")

## 📚 Next Steps

After completing this lab:

1. **Review all labs**: Consolidate your DuckDB knowledge
2. **Practice real projects**: Build applications using your preferred API
3. **Explore advanced features**: Look into specific language features
4. **Contribute to community**: Share your experiences and code